# RenderForge TTS — Audio Generator

Generate narration audio for RenderForge videos using **Qwen3-TTS**.

### Three ways to create voices
- **Step 4 — Voice Design** (1.7B): Describe the voice you want in natural language
- **Step 4c — Voice Cloning**: Upload a WAV/MP3 sample (3s minimum) and clone it
- **Step 4d — Custom Voice**: Pick a built-in speaker preset + control emotion/style

### IMPORTANT: Enable GPU
**Runtime → Change runtime type → A100 GPU → Save**

> A100 needed for VoiceDesign (1.7B). Cloning and CustomVoice use the lighter 0.6B model.

## Step 1: Setup

In [ ]:
!pip install -q qwen-tts soundfile numpy

import torch
import soundfile as sf
import numpy as np
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display
import os, json, time

os.makedirs("audio", exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Ready!")

## Step 2: Upload scripts.json

Generated locally by: `npx tsx content/generate-scripts.ts --limit 10`

In [ ]:
from google.colab import files

print("Upload scripts.json:")
uploaded = files.upload()
scripts = json.loads(list(uploaded.values())[0].decode('utf-8'))

total_sections = sum(len(s['sections']) for s in scripts)
print(f"\nLoaded {len(scripts)} posts, {total_sections} audio sections to generate")
for s in scripts[:5]:
    print(f"  [{s['postId']}] {s['template']} — {s['title'][:50]}")
if len(scripts) > 5:
    print(f"  ... and {len(scripts) - 5} more")

## Step 3: Load Models

Two models:
- **VoiceDesign** (1.7B) — creates voices from text descriptions
- **Base** (0.6B) — clones voices for fast, consistent generation

In [ ]:
print("Loading models...")

voice_design = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0", dtype=torch.bfloat16
)

tts = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0", dtype=torch.bfloat16
)

print("Models loaded!")

## Step 4: Create Voice

Design the narrator voice using natural language, then lock it in for consistent cloning.

**Modify the description below** to change the voice character.

In [ ]:
# ═══════════════════════════════════════════
#  VOICE DESIGN — describe your ideal narrator
# ═══════════════════════════════════════════

VOICE_DESCRIPTION = """
    Male, late 20s, confident and energetic. Motivational speaker tone.
    Clear American accent, medium-fast pace, slightly deep pitch.
    Sounds like a successful young entrepreneur giving real advice.
    Engaging and natural, not robotic. Slight intensity and urgency.
"""

# Sample text to generate the voice (should match channel tone)
VOICE_SAMPLE = "Your last dollar could be the first dollar of your empire. Stop making excuses and start building."

LANGUAGE = "English"

# ═══════════════════════════════════════════

print("Designing voice...")
wavs, sr = voice_design.generate_voice_design(
    text=VOICE_SAMPLE,
    language=LANGUAGE,
    instruct=VOICE_DESCRIPTION
)

# Create clone prompt for consistent voice across all lines
voice_prompt = tts.create_voice_clone_prompt(
    ref_audio=(wavs[0], sr),
    ref_text=VOICE_SAMPLE
)

sf.write("audio/voice_sample.wav", wavs[0], sr)
sample_duration = len(wavs[0]) / sr

print(f"Voice created! Sample: {sample_duration:.1f}s")
print("\nListen to your narrator:")
display(Audio(wavs[0], rate=sr))

print("\nHappy with the voice? Continue to Step 5.")
print("Want a different voice? Change VOICE_DESCRIPTION above and re-run this cell.")

## Step 4b: (Optional) Try Alternative Voices

Run this cell to audition different voice styles before committing.

In [ ]:
# Uncomment the voice style you want to try, or write your own

ALTERNATIVES = {
    "motivational_deep": """
        Male, 40s, deep resonant voice. Calm authority with motivational energy.
        Like a successful CEO sharing wisdom. Clear, powerful, measured pace.
    """,
    "young_energetic": """
        Male, early 20s, high energy and excitement. Fast-paced, enthusiastic.
        Like a hyped-up YouTuber who genuinely believes in what he's saying.
    """,
    "calm_stoic": """
        Male, 50s, calm and deliberate. Deep, wise voice with gravitas.
        Speaks slowly with purpose. Like Marcus Aurelius reading his journal.
    """,
    "kids_narrator": """
        Female, 20s, cheerful and playful. Bright, warm, slightly high pitch.
        Fun and engaging like a children's show host. Expressive and animated.
    """,
}

# Change this to try different voices
TRY_VOICE = "motivational_deep"

test_text = "Stop making excuses. Start making money. The grind doesn't stop."

print(f"Testing: {TRY_VOICE}")
wavs, sr = voice_design.generate_voice_design(
    text=test_text,
    language=LANGUAGE,
    instruct=ALTERNATIVES[TRY_VOICE]
)
display(Audio(wavs[0], rate=sr))

# If you like this voice, lock it in:
USE_THIS = False  # Set to True and re-run to use this voice
if USE_THIS:
    voice_prompt = tts.create_voice_clone_prompt(
        ref_audio=(wavs[0], sr),
        ref_text=test_text
    )
    print("Voice locked in! Continue to Step 5.")

## Step 4c: Voice Cloning from Audio Sample

Clone any voice with just 3 seconds of audio. Upload your best motivational voice WAV file and the Base model will replicate it.

**Use Step 4 OR 4c** — both set `voice_prompt` for Step 5.

In [ ]:
# Upload your reference voice audio
from google.colab import files

print("Upload your reference voice (WAV/MP3, 3-30s of clean speech):")
uploaded = files.upload()

ref_filename = list(uploaded.keys())[0]
# Save to audio/ directory
ref_audio_path = f"audio/{ref_filename}"
with open(ref_audio_path, "wb") as f:
    f.write(list(uploaded.values())[0])

print(f"\nUploaded: {ref_filename}")
print("Listen to your reference:")
display(Audio(ref_audio_path))

In [ ]:
# ═══════════════════════════════════════════
#  VOICE CLONING — clone from uploaded audio
# ═══════════════════════════════════════════

# Transcript of what the speaker says in the reference audio
# (accurate transcript = better voice match)
ref_text = """Your reference audio transcript goes here. Type exactly what the speaker says."""

LANGUAGE = "English"

# Load the Base model for voice cloning (skip if already loaded in Step 3)
if 'tts' not in dir():
    print("Loading Base model for voice cloning...")
    tts = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
        device_map="cuda:0", dtype=torch.bfloat16
    )
    print("Clone model loaded!")

# Generate test samples with the cloned voice
test_lines = [
    "Your last dollar could be the first dollar of your empire. Stop making excuses and start building.",
    "The grind doesn't care about your feelings. Show up every single day.",
    "While they sleep, you work. While they party, you study. That's the difference.",
]

print(f"\nCloning voice from: {ref_audio_path}")
print(f"Reference transcript: \"{ref_text[:60]}...\"\n")

for i, text in enumerate(test_lines, 1):
    wavs, sr = tts.generate_voice_clone(
        text=text,
        language=LANGUAGE,
        ref_audio=ref_audio_path,
        ref_text=ref_text,
    )

    filename = f"audio/clone_test_{i}.wav"
    sf.write(filename, wavs[0], sr)
    duration = len(wavs[0]) / sr
    print(f"Sample {i} ({duration:.1f}s): \"{text[:60]}...\"")
    display(Audio(wavs[0], rate=sr))
    print()

# Lock in this voice for batch generation (Step 5)
voice_prompt = tts.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=ref_text
)
print("Voice locked in! voice_prompt is set for Step 5.")
print("Not happy? Re-upload a different audio file and re-run.")

## Step 4d: (Optional) Custom Voice with Emotion Control

Use the **CustomVoice** model (0.6B) for built-in speaker presets with emotion/style instructions.
No reference audio needed — just pick a speaker and control the tone.

In [ ]:
# Load the CustomVoice model
print("Loading CustomVoice model...")
custom_voice = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    device_map="cuda:0", dtype=torch.bfloat16
)
print("CustomVoice model loaded!")

# Check available speakers and languages
print("\nAvailable speakers:", custom_voice.get_supported_speakers())
print("Supported languages:", custom_voice.get_supported_languages())

In [ ]:
# ═══════════════════════════════════════════
#  CUSTOM VOICE — pick a speaker + emotion
# ═══════════════════════════════════════════

# Pick a speaker from the list above
SPEAKER = "Chelsie"  # Change to any available speaker

# Emotion/style instructions — this is the power of CustomVoice
EMOTION_STYLES = {
    "motivational": "Speak with intense passion and conviction. Urgent, energetic, like motivating a crowd.",
    "calm_wisdom": "Speak slowly with deep wisdom. Calm, measured, like sharing a profound truth.",
    "excited": "Speak with high energy and excitement. Fast-paced, enthusiastic, uplifting.",
    "serious": "Speak with gravity and seriousness. Deliberate pace, weight behind every word.",
    "warm": "Speak warmly and encouragingly. Like a mentor giving genuine advice to someone they care about.",
}

STYLE = "motivational"  # Change to try different emotions
LANGUAGE = "English"

test_text = "Your last dollar could be the first dollar of your empire. Stop making excuses and start building."

print(f"Speaker: {SPEAKER} | Style: {STYLE}")
print(f"Text: \"{test_text[:60]}...\"\n")

wavs, sr = custom_voice.generate(
    text=test_text,
    speaker=SPEAKER,
    language=LANGUAGE,
    instruct=EMOTION_STYLES[STYLE],
)

sf.write("audio/custom_voice_test.wav", wavs[0], sr)
duration = len(wavs[0]) / sr
print(f"Generated: {duration:.1f}s")
display(Audio(wavs[0], rate=sr))

# Try all styles with the same speaker
print("\n--- Auditioning all emotion styles ---\n")
for style_name, instruct in EMOTION_STYLES.items():
    wavs, sr = custom_voice.generate(
        text=test_text,
        speaker=SPEAKER,
        language=LANGUAGE,
        instruct=instruct,
    )
    duration = len(wavs[0]) / sr
    print(f"{style_name} ({duration:.1f}s):")
    display(Audio(wavs[0], rate=sr))
    print()

## Step 5: Generate All Audio

Uses the cloned voice (0.6B Base) for fast, consistent generation.

> **Prerequisite:** Run **one** of Step 4, 4c, or 4d to set `voice_prompt` before running this.

In [ ]:
total = sum(len(s['sections']) for s in scripts)
done = 0
failed = []
start_time = time.time()

# Pause durations between sections (seconds)
PAUSE = 0.3
DRAMATIC_PAUSE = 0.6  # after intro and last slide

print(f"Generating {total} audio files for {len(scripts)} posts...")
print("=" * 60)

for post in scripts:
    post_id = post['postId']
    post_dir = os.path.join("audio", post_id)
    os.makedirs(post_dir, exist_ok=True)

    print(f"\n[{post_id}] {post['template']} — {post['title'][:50]}")

    post_wavs = []
    section_sr = None

    for i, section in enumerate(post['sections']):
        done += 1
        elapsed = time.time() - start_time
        per_item = elapsed / done if done > 0 else 0
        eta = per_item * (total - done)
        eta_str = f"{int(eta // 60)}m {int(eta % 60)}s" if eta > 60 else f"{int(eta)}s"

        output_path = os.path.join("audio", section['audioFile'])
        text = section['text']

        print(f"  [{done}/{total}] {section['key']}: \"{text[:55]}{'...' if len(text) > 55 else ''}\" (ETA: {eta_str})", end="")

        try:
            wavs, current_sr = tts.generate_voice_clone(
                text=text,
                language=LANGUAGE,
                voice_clone_prompt=voice_prompt
            )

            sf.write(output_path, wavs[0], current_sr)
            duration = len(wavs[0]) / current_sr
            print(f" -> {duration:.1f}s")

            post_wavs.append(wavs[0])
            if section_sr is None:
                section_sr = current_sr
        except Exception as e:
            print(f" FAILED: {str(e)[:80]}")
            failed.append({'post': post_id, 'section': section['key'], 'error': str(e)})

    # Combine into full narration with pauses
    if post_wavs and section_sr:
        pause = np.zeros(int(section_sr * PAUSE), dtype=post_wavs[0].dtype)
        dramatic = np.zeros(int(section_sr * DRAMATIC_PAUSE), dtype=post_wavs[0].dtype)

        parts = []
        for j, wav in enumerate(post_wavs):
            parts.append(wav)
            if j < len(post_wavs) - 1:
                is_dramatic = (j == 0) or (j == len(post_wavs) - 2)
                parts.append(dramatic if is_dramatic else pause)

        full_wav = np.concatenate(parts)
        sf.write(os.path.join(post_dir, "full.wav"), full_wav, section_sr)
        print(f"  >> Full: {len(full_wav) / section_sr:.1f}s")

total_time = time.time() - start_time
print("\n" + "=" * 60)
print(f"DONE! {done - len(failed)}/{total} files in {int(total_time // 60)}m {int(total_time % 60)}s")
if failed:
    print(f"\nFailed ({len(failed)}):")
    for f in failed:
        print(f"  {f['post']}/{f['section']}: {f['error'][:80]}")

## Step 6: Preview

In [ ]:
# Preview first post
first = scripts[0]
full_path = f"audio/{first['postId']}/full.wav"

if os.path.exists(full_path):
    print(f"[{first['postId']}] {first['title']}")
    print(f"Full narration:")
    display(Audio(full_path))

    print(f"\nSections:")
    for sec in first['sections']:
        sec_path = f"audio/{sec['audioFile']}"
        if os.path.exists(sec_path):
            print(f"  {sec['key']}: \"{sec['text'][:60]}\"")
            display(Audio(sec_path))
else:
    print("No audio yet. Run Step 5 first.")

## Step 7: Download

Then locally:
```bash
# Extract zip so files are at content/audio/day01-post1/intro.wav
npx tsx content/audio-sync.ts --limit 10
# Output: output/synced/story/*.mp4
```

In [ ]:
import shutil

# Count files
total_size = 0
file_count = 0
for root, dirs, fnames in os.walk("audio"):
    for f in fnames:
        if f.endswith('.wav'):
            total_size += os.path.getsize(os.path.join(root, f))
            file_count += 1

print(f"Audio files: {file_count}")
print(f"Total size:  {total_size / 1024 / 1024:.1f} MB")

# Zip and download
shutil.make_archive("renderforge-audio", "zip", ".", "audio")
zip_size = os.path.getsize("renderforge-audio.zip") / 1024 / 1024
print(f"\nZip: renderforge-audio.zip ({zip_size:.1f} MB)")

try:
    from google.colab import files
    files.download("renderforge-audio.zip")
except:
    print("Download 'renderforge-audio.zip' from the files panel")

print("\nNext: extract into renderforge/content/ and run audio-sync.ts")

---

## Voice Presets

Copy-paste these into `VOICE_DESCRIPTION` in Step 4:

| Channel | Description |
|---------|-------------|
| **Your Last Dollar** | `Male, late 20s, confident and energetic. Motivational speaker tone. Clear American accent, medium-fast pace, slightly deep. Like a young entrepreneur giving real advice.` |
| **KnockKnockZoo** | `Female, 20s, cheerful and playful. Bright, warm, slightly high pitch. Fun and engaging like a children's show host. Expressive and animated.` |
| **Stoicism/Wisdom** | `Male, 50s, calm and deliberate. Deep, wise voice with gravitas. Speaks slowly with purpose. Like Marcus Aurelius reading his journal.` |
| **Horror/Creepy** | `Male, 30s, quiet and whispery. Unsettlingly calm. Slow pace with dramatic pauses. Every word carries weight and dread.` |
| **Tech/AI News** | `Male, 30s, clear and professional. Neutral informative tone, medium pace. Like a tech journalist breaking news.` |
| **Dubai Luxury** | `Male, 40s, smooth and sophisticated. Slight British accent, elegant and refined. Medium-slow pace. Like narrating a luxury brand film.` |

---

## Best Voices to Clone for Motivation (Step 4c)

Search YouTube/podcasts for 10-20s clean clips to upload in Step 4c.

### Top Motivational Voices
| Voice | Style | Where to find clean samples |
|-------|-------|----------------------------|
| **Les Brown** | Deep, powerful, passionate. Gold standard motivational energy. | YouTube: "Les Brown motivational speech" |
| **Eric Thomas (ET)** | Raw intensity, street-smart. "When you want to succeed..." | YouTube: "ET The Hip Hop Preacher" |
| **David Goggins** | Gritty, no-nonsense, aggressive. Pure discipline. | Podcast clips, YouTube interviews |
| **Tony Robbins** | Commanding, high-energy, authoritative. | YouTube seminars, TED talks |
| **Denzel Washington** | Smooth authority with gravitas. Calm but powerful. | Commencement speeches on YouTube |

### Narration / Versatile Voices
| Voice | Style | Where to find clean samples |
|-------|-------|----------------------------|
| **Morgan Freeman** | Warm, deep, trustworthy. The narrator voice. | Documentary narrations, interviews |
| **Idris Elba** | Smooth, deep, British sophistication with edge. | Interviews, audiobook samples |
| **Matthew McConaughey** | Laid-back confidence, philosophical. | Speeches, podcast appearances |

### Female Motivational
| Voice | Style | Where to find clean samples |
|-------|-------|----------------------------|
| **Lisa Nichols** | Powerful, emotional, transformational. | YouTube speeches, Mindvalley talks |
| **Mel Robbins** | Direct, energetic, no-BS. Practical motivation. | YouTube, podcast clips |

### Tips for Clean Clone Samples
1. **Commencement speeches** — clean audio, single speaker, no music
2. **Podcast intros** — often the cleanest audio quality
3. **Trim to 10-20s** of the best, most characteristic section
4. **Avoid** clips with background music, audience noise, echo, or multiple speakers
5. Use **Audacity** (free) to trim, denoise, and export as WAV